# lets_route — `route_f` Playground

This notebook runs on the **`route_f`** kernel (Python 3.12, uv-managed).
The same environment is used in the terminal via:

```bash
source .route_f/bin/activate
```

> If the kernel isn't listed in VS Code yet, finish the setup first:
> `./setup_route_f.sh` (or verify with `jupyter kernelspec list`).

In [1]:
import sys, platform
print(f"Python : {sys.version.split()[0]} (executable: {sys.executable})")
print(f"Platform: {platform.platform()}")

try:
    import torch
    print(f"torch : {torch.__version__}")
    print(f"CUDA available : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU : {torch.cuda.get_device_name(0)}")
    try:
        torch.zeros(1, device=torch.cuda.is_available() and 'cuda' or 'cpu')
        print("tensor allocation ok")
    except Exception as e:
        print("tensor allocation failed:", e)
except ImportError:
    print("torch not installed yet — run ./setup_route_f.sh")

Python : 3.12.13 (executable: /home/mr/Desktop/Fireworks/.route_f/bin/python)
Platform: Linux-7.0.0-38-generic-x86_64-with-glibc2.39
torch : 2.15.0+cu132
CUDA available : True
GPU : NVIDIA GeForce RTX 3060 Laptop GPU
tensor allocation ok


In [2]:
# Quick sanity check of the training stack
import importlib
mods = ["transformers", "datasets", "accelerate", "peft", "pandas", "numpy", "pyarrow"]
for m in mods:
    try:
        mod = importlib.import_module(m)
        print(f"{m:16s} {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"{m:16s} NOT INSTALLED")

/home/mr/Desktop/Fireworks/.route_f/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers     5.22.1
datasets         5.1.0
accelerate       1.15.0
peft             0.21.2
pandas           3.1.0
numpy            2.5.4
pyarrow          26.0.0


## Dataset: xRouteBench (`ulab-ai/xRouteBench`)

LLM-routing benchmark — each row is a `(query, candidate model)` pair with the
model's response, token usage, latency, and a `performance` score. A router
learns which model to pick per query, trading off performance vs price.

- **Configs (scenarios):** `llmrouter_generic`, `memory_locomo`, `memory_longmemeval`,
  `timeseries`, `video`, `multimodal_geometry3k`, `multimodal_mathvista`, `personalized`.
- **Raw queries:** `<scenario>_queries` (train/valid/test) — no model executions.
- **Pricing pool:** `llm_candidates` (18 models with prices).
- **No bulk download:** the cells below inspect everything via the HF API and
  stream only a few rows (parquet pulled lazily). Set upstream=False`` only when
  you actually want to download the full dataset for training.

In [3]:
import sys
sys.path.insert(0, ".")
from data.load_xroutebench import list_configs, schema, peek

print("Available configs (from HF datasets-server API, no download):")
for c in list_configs():
    print("  -", c)

cols, samples = schema("llmrouter_generic", "train")
print("\nSchema for llmrouter_generic/train:")
print(cols)

print("\nFirst 2 rows via STREAMING (only the first parquet chunk is read):")
for row in peek("llmrouter_generic", "train", n=2):
    print("  -", {k: row[k] for k in ("task_name", "model_name", "metric")})

Available configs (from HF datasets-server API, no download):
  - llm_candidates
  - llmrouter_generic
  - llmrouter_generic_queries
  - memory_locomo
  - memory_locomo_queries
  - memory_longmemeval
  - memory_longmemeval_queries
  - multimodal_geometry3k
  - multimodal_geometry3k_queries
  - multimodal_mathvista
  - multimodal_mathvista_queries
  - personalized
  - personalized_queries
  - timeseries
  - timeseries_queries
  - video
  - video_queries

Schema for llmrouter_generic/train:
['task_name', 'query', 'ground_truth', 'metric', 'choices', 'task_id', 'model_name', 'response', 'token_num', 'input_tokens', 'output_tokens', 'response_time', 'performance', 'embedding_id']

First 2 rows via STREAMING (only the first parquet chunk is read):
  - {'task_name': 'mbpp', 'model_name': 'qwen2.5-7b-instruct', 'metric': 'code_eval'}
  - {'task_name': 'mbpp', 'model_name': 'gemma-2-9b-it', 'metric': 'code_eval'}


In [ ]:
# Download the pinned official Qwen3.5-0.8B checkpoint into this repo.
from pathlib import Path
from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen3.5-0.8B"
MODEL_REVISION = "2fc06364715b967f1860aea9cf38778875588b17"
repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "route_f_playground.ipynb").exists()
)
MODEL_DIR = repo_root / "models" / "qwen3.5-0.8b"

snapshot_path = snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_REVISION,
    local_dir=MODEL_DIR,
)
downloaded_bytes = sum(
    path.stat().st_size for path in MODEL_DIR.rglob("*") if path.is_file()
)
print(f"Downloaded {MODEL_ID}@{MODEL_REVISION[:8]}")
print(f"Location: {snapshot_path}")
print(f"Size: {downloaded_bytes / 1024**3:.2f} GiB")

In [ ]:
# SGLang is the only inference backend. Run the server in a separate terminal/runtime.
import json
import urllib.error
import urllib.request

SGLANG_HOST = "127.0.0.1"
SGLANG_PORT = 30000
SGLANG_BASE_URL = f"http://{SGLANG_HOST}:{SGLANG_PORT}/v1"
SGLANG_MODEL_NAME = "qwen3.5-0.8b"
SGLANG_API_KEY = "EMPTY"
SGLANG_LAUNCH_COMMAND = (
    f'python -m sglang.launch_server --model-path "{MODEL_DIR}" '
    f'--served-model-name {SGLANG_MODEL_NAME} --host {SGLANG_HOST} --port {SGLANG_PORT} '
    '--device cuda --dtype bfloat16 --mem-fraction-static 0.70 '
    '--enable-deterministic-inference'
)

def sglang_json(method, path, payload=None, timeout=120):
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        f"{SGLANG_BASE_URL}{path}",
        data=body,
        method=method,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {SGLANG_API_KEY}",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.load(response)
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"SGLang HTTP {exc.code}: {detail}") from exc

try:
    model_catalog = sglang_json("GET", "/models", timeout=10)
    served_models = [item["id"] for item in model_catalog.get("data", [])]
    SGLANG_CONNECTED = SGLANG_MODEL_NAME in served_models
except (OSError, RuntimeError):
    served_models, SGLANG_CONNECTED = [], False
if SGLANG_CONNECTED:
    print(f"Connected to SGLang at {SGLANG_BASE_URL}; models={served_models}")
else:
    print("SGLang is not connected. Start it in a separate GPU terminal with:")
    print(SGLANG_LAUNCH_COMMAND)

## Plug-and-play LLM evaluation on xRouteBench Parquet

The loader below validates the full query-by-model matrix, returns one row per query instance, and keeps ground truth separate from the messages sent to an LLM. Inference goes only through an SGLang OpenAI-compatible endpoint, so models can be swapped by changing the served model without changing sampling or scoring.

This first check uses 12 deterministic multiple-choice queries because their `em_mc` ground truth can be scored exactly. The notebook never loads model weights directly; SGLang owns GPU execution and the notebook measures end-to-end API latency and token usage.

In [ ]:
import hashlib
import re
import time
from pathlib import Path
import numpy as np
import pandas as pd

ROUTING_COLUMNS = [
    "embedding_id", "task_name", "query", "ground_truth",
    "metric", "model_name", "performance", "input_tokens",
    "output_tokens", "response_time",
]

def load_xroute_queries(repo_root, split="test"):
    """Return one validated row per query and the complete long matrix."""
    path = Path(repo_root) / f"data/raw/llmrouter_generic/{split}-000.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Missing xRouteBench Parquet: {path}")
    frame = pd.read_parquet(path, columns=ROUTING_COLUMNS)
    missing = set(ROUTING_COLUMNS) - set(frame.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    if frame.duplicated(["embedding_id", "model_name"]).any():
        raise ValueError("Duplicate (embedding_id, model_name) executions found")

    candidate_count = frame["model_name"].nunique()
    per_query = frame.groupby("embedding_id", sort=False).agg(
        rows=("model_name", "size"),
        models=("model_name", "nunique"),
        queries=("query", "nunique"),
        tasks=("task_name", "nunique"),
        metrics=("metric", "nunique"),
    )
    invalid = per_query[
        (per_query["rows"] != candidate_count)
        | (per_query["models"] != candidate_count)
        | (per_query[["queries", "tasks", "metrics"]] != 1).any(axis=1)
    ]
    if len(invalid):
        raise ValueError(f"{len(invalid)} incomplete or inconsistent query matrices")

    query_table = (
        frame.sort_values("embedding_id")
        .drop_duplicates("embedding_id")
        [["embedding_id", "task_name", "query", "ground_truth", "metric"]]
        .reset_index(drop=True)
    )
    query_table["query_hash"] = query_table["query"].map(
        lambda value: hashlib.sha256(str(value).encode("utf-8")).hexdigest()
    )
    audit = {
        "split": split,
        "execution_rows": len(frame),
        "query_instances": len(query_table),
        "unique_prompt_texts": query_table["query_hash"].nunique(),
        "candidate_models": candidate_count,
    }
    return query_table, frame, audit

def select_mc_queries(query_table, tasks, per_task=2, seed=42):
    """Deterministically sample exact-score multiple-choice queries by task."""
    selections = []
    for task in tasks:
        task_rows = query_table[
            query_table["task_name"].eq(task) & query_table["metric"].eq("em_mc")
        ].copy()
        if len(task_rows) < per_task:
            raise ValueError(f"Task {task!r} has only {len(task_rows)} em_mc queries")
        task_rows["sample_order"] = task_rows["query_hash"].map(
            lambda value: hashlib.sha256(f"{seed}:{value}".encode()).hexdigest()
        )
        selections.append(task_rows.sort_values("sample_order").head(per_task))
    selected = pd.concat(selections, ignore_index=True).drop(columns="sample_order")
    if len(selected) < 10:
        raise ValueError("Evaluation must contain at least 10 queries")
    return selected

def xroute_messages(raw_query):
    """Convert xRouteBench's combined prompt into proper chat roles."""
    raw_query = str(raw_query)
    system_marker, user_marker = "[System Instruction]", "[User Query]"
    if raw_query.startswith(system_marker) and user_marker in raw_query:
        system_text, user_text = raw_query[len(system_marker):].split(user_marker, 1)
        return [
            {"role": "system", "content": system_text.strip()},
            {"role": "user", "content": user_text.strip()},
        ]
    return [{"role": "user", "content": raw_query}]

def extract_mc_choice(response):
    """Extract a declared A-E answer without rewarding option mentions in prose."""
    text = str(response).strip().upper()
    standalone = re.fullmatch(r"\s*\(?([A-E])\)?(?:[.)])?\s*", text)
    if standalone:
        return standalone.group(1)
    declared = re.findall(
        r"(?:FINAL|CORRECT)\s+(?:ANSWER|OPTION)(?:\s+IS)?[:\s]+\(?([A-E])\)?(?:\b|$)",
        text,
    )
    if declared:
        return declared[-1]
    last_line = next((line.strip() for line in reversed(text.splitlines()) if line.strip()), "")
    final_line = re.fullmatch(
        r"(?:ANSWER[:\s]+)?\(?([A-E])\)?[.)]?(?:\s+.+)?", last_line
    )
    if final_line:
        return final_line.group(1)
    leading = re.match(r"^\s*([A-E])(?:[.)]|\s*[-:])", text)
    if leading:
        return leading.group(1)
    return None

def evaluate_generator(queries, generate_fn):
    """Evaluate any adapter returning response/tokens/latency/cost fields."""
    rows = []
    for position, row in enumerate(queries.itertuples(index=False), start=1):
        try:
            result = generate_fn(xroute_messages(row.query))
            parsed = extract_mc_choice(result["response"])
            error = None
        except Exception as exc:
            result = {"response": "", "input_tokens": 0, "output_tokens": 0, "latency_s": np.nan, "cost_usd": np.nan}
            parsed, error = None, f"{type(exc).__name__}: {exc}"
        expected = str(row.ground_truth).strip().upper()
        rows.append({
            "embedding_id": int(row.embedding_id),
            "task_name": row.task_name,
            "expected": expected,
            "parsed_answer": parsed,
            "correct": bool(parsed == expected),
            "response": result.get("response", ""),
            "input_tokens": result.get("input_tokens", np.nan),
            "output_tokens": result.get("output_tokens", np.nan),
            "latency_s": result.get("latency_s", np.nan),
            "cost_usd": result.get("cost_usd", np.nan),
            "finish_reason": result.get("finish_reason"),
            "error": error,
        })
        print(f"[{position:02d}/{len(queries)}] {row.task_name}: expected={expected}, predicted={parsed}")
    return pd.DataFrame(rows)

In [ ]:
EVAL_TASKS = ("mmlu", "arc_challenge", "openbook_qa", "commonsense_qa", "boolq", "hellaswag")
query_table, test_long, loader_audit = load_xroute_queries(repo_root, split="test")
eval_queries = select_mc_queries(query_table, EVAL_TASKS, per_task=2, seed=42)
print(loader_audit)
print("\nEvaluation sample:")
print(eval_queries[["embedding_id", "task_name", "ground_truth"]].to_string(index=False))

In [ ]:
# Any SGLang-served model can use this adapter. Set prices for hosted endpoints,
# or an amortized server hourly rate for a locally hosted GPU.
SGLANG_INPUT_PRICE_PER_1M = None
SGLANG_OUTPUT_PRICE_PER_1M = None
SGLANG_SERVER_USD_PER_HOUR = None
EVAL_MAX_TOKENS = 512
model_catalog = sglang_json("GET", "/models", timeout=10)
served_models = [item["id"] for item in model_catalog.get("data", [])]
if SGLANG_MODEL_NAME not in served_models:
    raise RuntimeError(f"Expected {SGLANG_MODEL_NAME!r}; server exposes {served_models}")

def generate_with_sglang(messages):
    payload = {
        "model": SGLANG_MODEL_NAME,
        "messages": messages,
        "temperature": 0,
        "max_tokens": EVAL_MAX_TOKENS,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    start = time.perf_counter()
    data = sglang_json("POST", "/chat/completions", payload, timeout=180)
    latency_s = time.perf_counter() - start
    choice = data["choices"][0]
    usage = data.get("usage") or {}
    input_tokens = int(usage.get("prompt_tokens", 0))
    output_tokens = int(usage.get("completion_tokens", 0))
    if SGLANG_INPUT_PRICE_PER_1M is not None and SGLANG_OUTPUT_PRICE_PER_1M is not None:
        cost_usd = (
            input_tokens * SGLANG_INPUT_PRICE_PER_1M
            + output_tokens * SGLANG_OUTPUT_PRICE_PER_1M
        ) / 1_000_000.0
    elif SGLANG_SERVER_USD_PER_HOUR is not None:
        cost_usd = latency_s * SGLANG_SERVER_USD_PER_HOUR / 3600.0
    else:
        cost_usd = np.nan
    return {
        "response": (choice.get("message") or {}).get("content", "") or "",
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "latency_s": latency_s,
        "cost_usd": cost_usd,
        "finish_reason": choice.get("finish_reason"),
    }

sglang_eval = evaluate_generator(eval_queries, generate_with_sglang)
sglang_accuracy = sglang_eval["correct"].mean()
sglang_tokens_per_s = sglang_eval["output_tokens"].sum() / sglang_eval["latency_s"].sum()
print("\nPer-query results:")
sglang_display = sglang_eval[["embedding_id", "task_name", "expected", "parsed_answer", "correct", "finish_reason", "latency_s"]].copy()
sglang_display["response_preview"] = sglang_eval["response"].str.replace("\n", " ", regex=False).str.slice(0, 160)
print(sglang_display.to_string(index=False))
print(f"\nSGLang model accuracy: {sglang_accuracy:.1%} ({sglang_eval['correct'].sum()}/{len(sglang_eval)})")
print(f"Mean API latency: {sglang_eval['latency_s'].mean():.3f}s | Output throughput: {sglang_tokens_per_s:.2f} tokens/s")
print(f"Length-truncated responses: {(sglang_eval['finish_reason'] == 'length').sum()}/{len(sglang_eval)}")
if sglang_eval["cost_usd"].isna().all():
    print("Dollar cost: configure token prices or SGLANG_SERVER_USD_PER_HOUR.")
else:
    print(f"Cost per 1,000 queries: ${sglang_eval['cost_usd'].mean() * 1000:.4f}")

In [ ]:
# Compare the SGLang-served model with recorded xRouteBench candidates on the same IDs.
candidate_prices = pd.read_parquet(
    repo_root / "data/raw/llm_candidates/train-000.parquet"
)[["model_name", "input_price_per_1m", "output_price_per_1m"]]
selected_ids = set(eval_queries["embedding_id"])
candidate_eval = test_long[test_long["embedding_id"].isin(selected_ids)].merge(
    candidate_prices, on="model_name", how="left", validate="many_to_one"
)
candidate_eval["cost_usd"] = (
    candidate_eval["input_tokens"] * candidate_eval["input_price_per_1m"]
    + candidate_eval["output_tokens"] * candidate_eval["output_price_per_1m"]
) / 1_000_000.0
candidate_summary = (
    candidate_eval.groupby("model_name")
    .agg(
        accuracy=("performance", "mean"),
        mean_cost_per_1k_usd=("cost_usd", lambda values: values.mean() * 1000),
        mean_latency_s=("response_time", "mean"),
    )
    .reset_index()
)
sglang_summary = pd.DataFrame([{
    "model_name": f"{SGLANG_MODEL_NAME} (SGLang)",
    "accuracy": sglang_accuracy,
    "mean_cost_per_1k_usd": sglang_eval["cost_usd"].mean() * 1000,
    "mean_latency_s": sglang_eval["latency_s"].mean(),
}])
comparison = pd.concat([sglang_summary, candidate_summary], ignore_index=True)
comparison = comparison.sort_values(["accuracy", "mean_cost_per_1k_usd"], ascending=[False, True])
print(comparison.to_string(index=False, na_rep="not configured", float_format=lambda value: f"{value:.6f}"))
cheapest_recorded = candidate_summary.loc[candidate_summary["mean_cost_per_1k_usd"].idxmin()]
break_even_hourly_rate = (
    cheapest_recorded["mean_cost_per_1k_usd"] * 3600
    / (sglang_eval["latency_s"].mean() * 1000)
)
print(
    f"\nSequential break-even vs {cheapest_recorded['model_name']}: "
    f"local GPU must cost less than ${break_even_hourly_rate:.4f}/hour."
)

### SGLang runtime boundary

The earlier direct-Transformers model loading and GPU generation path has been removed. The notebook is now only a dataset loader, evaluator, and OpenAI-compatible client. SGLang owns model loading, tokenization, batching, KV cache, and GPU execution.

Start the server in a separate GPU environment using the command printed by `connect-sglang-server`. The explicit `--device cuda` flag prevents CPU fallback. Do not install SGLang into `route_f`: the current SGLang resolver would replace PyTorch, Transformers, NumPy, Hugging Face Hub, and multiple CUDA libraries. Keeping server and evaluation environments separate makes model backends genuinely plug-and-play.

To swap models, launch another SGLang server and change `SGLANG_BASE_URL` and `SGLANG_MODEL_NAME`. The loader, deterministic sample, scorer, cost accounting, and xRouteBench comparison remain unchanged.

## Mercury 3
Uses `INCEPTION_API_KEY` from the local `.env` file. Running the next cell sends a request to Inception.

In [ ]:
from models.mercury import chat

response = chat([{"role": "user", "content": "What is a diffusion model?"}])
print(response["choices"][0]["message"]["content"])